# 「吉祥寺」に似た街はどこ？

東京都内の駅周辺を、同じ1km四方・同じPOI特徴で比較します。店の多さ、種類の構成、集まり方を組み合わせます。**似ているのは選んだPOI特徴だけ**で、景観や住みやすさの評価ではありません。

[固定設計](../docs/similar-regions-design.md) / [再現手順](../docs/similar-regions.md) / [検証](../docs/similar-regions-validation.md)

## 1. 問いと比較条件

N02-25の代表点が東京都内にある駅を、23区と多摩を含めて対象にします。EPSG:6677は東京周辺をメートルで扱う座標系です。固定release 2026-09-23.0の取得済みデータを使い、このNotebookからネットワーク取得は行いません。初回の入力準備は再現手順を確認してください。

In [ ]:
from geoai_open_lab.tokyo_region_features import run_comparison
from geoai_open_lab.similar_region_figures import save_similarity_figures
from IPython.display import SVG, display
profiles, rankings, stability, manifest, output_dir = run_comparison()
figures = save_similarity_figures(output_dir)

## 2. 同じ列へ変換

1地域が1行、固定15列です。吉祥寺は前回と同じ値になることを実行時に検証します。250m・500m格子は集計根拠です。

下は主要列の抜粋です。固定15列の全表は出力先の`region_features.csv`に保存します。

In [ ]:
print(f"対象駅数: {len(profiles)} / 吉祥寺の前回値との一致: {manifest['kichijoji_reconciled']}")
profiles.loc[profiles.region_id == "n02-003584", ["region_label", "poi_count", "poi_density_per_km2", "cafe_share", "peak_to_mean_density_250m"]]

## 3. 密度だけで探す

密度はlog1p変換して標準化します。吉祥寺自身と、吉祥寺の正方形と正の面積で重なる駅を除きます。

In [ ]:
rankings["density_only"].dropna(subset=["rank"]).head(10)[["rank", "station_name", "distance", "main_rank"]]

## 4. 地域特徴量で探す

密度と集中度はlog1p、各列を中央値・IQRで標準化。カテゴリ6列は各1/√6倍とし、3ブロックのユークリッド距離を使います。小さいほど近く、パーセントの「類似度」ではありません。

In [ ]:
display(SVG(filename=str(output_dir / "top10.svg")))
display(SVG(filename=str(output_dir / "candidate_map.svg")))

## 5. なぜ似ているか

生の数値を比べ、二乗距離のブロック別寄与も確認します。寄与率は「違いを生んだ割合」であり、似ている理由の確率ではありません。距離0では寄与率は未定義です。

In [ ]:
display(SVG(filename=str(output_dir / "profile_comparison.svg")))
rankings["main"].dropna(subset=["rank"]).head(3)[["station_name", "density_contribution", "composition_contribution", "concentration_contribution"]]

## 6. 条件を変えた安定性

密度のみ、標準化変更、密度除外、集中度除外を同じ候補で比較します。方式間で距離の尺度は異なるため、数値の小ささを直接比較しません。

In [ ]:
display(SVG(filename=str(output_dir / "rank_comparison.svg")))
display(SVG(filename=str(output_dir / "stability.svg")))
stability

## 7. 言えること・言えないこと

今回の主順位トップ3は町田・自由が丘・立川南。密度だけのトップ10とは共通0件で、通常標準化では8件が残りました。選ぶ特徴や方式で順位が変わります。

- POI登録を見ており、営業中店舗数や実際の人出ではありません。その他とカテゴリ欠損の差は残ります。
- 1km四方は公式の街域・商圏・駅の縄張りではありません。
- 井の頭公園は吉祥寺AOIと重なるため主順位から除外しています。
- 候補駅同士の重複は除外していません。立川南・立川北・立川は重なるAOIを持つ別駅グループで、独立した3つの街を発見した意味ではありません。
- 景観・人口・交通利便性・地価・住みやすさを含めていません。順位は街の優劣ではありません。
- 入力の完全性、説明可能性、条件変更への安定性を検証しており、正解ラベルによる精度は評価していません。

次回は同じ地域特徴量を使ったクラスタリングを検討します。今回クラスタリングやEmbeddingは行いません。